# Retail Sales — Exploratory Data Analysis

## 1. Business Context & Objectives
This project explores retail transactions to identify sales trends, customer behaviour, category performance, and actionable business insights.

**Tools:** Python 3.11.15, pandas, NumPy, Matplotlib, Seaborn, Jupyter Notebook.

**Project:** OASIS Infobyte Data Analytics — Level 1, Task 1.

## 2. Business Questions
1. How does revenue evolve over time?
2. Are there monthly or quarterly sales patterns?
3. Which categories generate the most revenue?
4. How does purchasing behaviour differ by gender?
5. How does transaction value vary across age groups?
6. Which customers contribute the most revenue?
7. How do categories differ in volume, revenue, average transaction value and profitability?
8. What relationships exist among age, quantity, price, COGS and sales?
9. Are there unusual transactions or outliers?
10. What additional pattern can support business decisions?

**Data limitation:** the dataset has `category` but no product identifier/name, so a true Top-10-products analysis is not possible. Category-level analysis is used instead.

## 3. Environment & Libraries

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Python version: {sys.version.split()[0]}")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", context="notebook")

## 4. Data Loading

In [ ]:
DATA_PATH = "../data/Retail_Sales.csv"
df_raw = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df_raw.shape}")
df_raw.head()

In [ ]:
df_raw.info()
df_raw.describe().T

## 5. Data Understanding & Quality Assessment

In [ ]:
df_raw.nunique().sort_values()

In [ ]:
print("Gender values:")
print(df_raw["gender"].value_counts(dropna=False))
print("\nCategory values:")
print(df_raw["category"].value_counts(dropna=False))
print("\nUnique transactions:", df_raw["transactions_id"].nunique())
print("Unique customers:", df_raw["customer_id"].nunique())

In [ ]:
missing_report = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_percentage": df_raw.isna().mean() * 100
}).sort_values("missing_count", ascending=False)
missing_report

In [ ]:
print("Duplicate rows:", df_raw.duplicated().sum())
raw_numeric_cols = ["age", "quantiy", "price_per_unit", "cogs", "total_sale"]
for col in raw_numeric_cols:
    print(f"{col}: {(df_raw[col] <= 0).sum()} non-positive values")

## 6. Data Preparation & Validation

In [ ]:
# Preserve the raw dataset; perform transformations on a separate working copy.
df_clean = df_raw.copy()
df_clean = df_clean.rename(columns={"quantiy": "quantity"})
df_clean["sale_date"] = pd.to_datetime(df_clean["sale_date"], errors="coerce")
df_clean.dtypes

### 6.1 Validate Total Sales
For complete records, **total sale = quantity × price per unit**. This validation checks the internal consistency of the sales measure.

In [ ]:
df_clean["calculated_total_sale"] = df_clean["quantity"] * df_clean["price_per_unit"]
df_clean["sales_difference"] = df_clean["total_sale"] - df_clean["calculated_total_sale"]
print("Inconsistent complete transactions:", df_clean["sales_difference"].abs().gt(0.01).sum())
reconstructable = df_clean["total_sale"].isna() & df_clean["quantity"].notna() & df_clean["price_per_unit"].notna()
df_clean.loc[reconstructable, "total_sale"] = df_clean.loc[reconstructable, "quantity"] * df_clean.loc[reconstructable, "price_per_unit"]
print("Reconstructed total_sale values:", reconstructable.sum())
print("Remaining missing total_sale values:", df_clean["total_sale"].isna().sum())

### 6.2 Missing-Value Treatment
The dataset contains 10 missing ages and three rows where the transaction-level numeric fields are missing together. Because those three rows cannot be reliably reconstructed from the available fields, they are retained but excluded from calculations requiring complete sales measures.

In [ ]:
df_clean[df_clean.isna().any(axis=1)]

In [ ]:
df_clean["year"] = df_clean["sale_date"].dt.year
df_clean["month"] = df_clean["sale_date"].dt.month
df_clean["quarter"] = df_clean["sale_date"].dt.to_period("Q").astype(str)
df_clean["profit"] = df_clean["total_sale"] - df_clean["cogs"]
bins = [0, 24, 34, 44, 54, 64, np.inf]
labels = ["Under 25", "25–34", "35–44", "45–54", "55–64", "65+"]
df_clean["age_group"] = pd.cut(df_clean["age"], bins=bins, labels=labels)

# 7. Exploratory Data Analysis
Each section combines quantitative summaries with visual evidence.

## 7.1 Descriptive Statistics

In [ ]:
numeric_cols = ["age", "quantity", "price_per_unit", "cogs", "total_sale", "profit"]
descriptive_stats = pd.DataFrame({
    "mean": df_clean[numeric_cols].mean(),
    "median": df_clean[numeric_cols].median(),
    "mode": df_clean[numeric_cols].mode().iloc[0],
    "std": df_clean[numeric_cols].std()
})
descriptive_stats

**Observation:** Revenue-related variables are right-skewed: total sales have a mean of $456.54 versus a median of $150.00, while profit has a mean of $361.52 versus a median of $106.50. A smaller number of high-value transactions therefore pulls the average upward. Quantity is much less dispersed (mean 2.51; median 3.00).

## 7.2 Monthly and Quarterly Sales Trends

In [ ]:
monthly_sales = (df_clean.dropna(subset=["sale_date", "total_sale"]).set_index("sale_date").resample("MS").agg(
    revenue=("total_sale", "sum"), transactions=("transactions_id", "count"), quantity_sold=("quantity", "sum"), profit=("profit", "sum")
).reset_index())
monthly_sales["average_transaction_value"] = monthly_sales["revenue"] / monthly_sales["transactions"]
monthly_sales

In [ ]:
plt.figure(figsize=(14, 6))
sns.lineplot(data=monthly_sales, x="sale_date", y="revenue", marker="o")
plt.title("Monthly Retail Sales Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Observation:** Monthly revenue varies substantially, with the strongest concentration toward the end of each year. The pattern is consistent with the quarterly results, but revenue should be interpreted together with transaction volume and average transaction value.

In [ ]:
quarterly_sales = (df_clean.dropna(subset=["sale_date", "total_sale"]).groupby("quarter").agg(
    revenue=("total_sale", "sum"), transactions=("transactions_id", "count"), quantity_sold=("quantity", "sum"), profit=("profit", "sum")
).reset_index())
quarterly_sales

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=quarterly_sales, x="quarter", y="revenue", marker="o")
plt.title("Quarterly Retail Sales Revenue")
plt.xlabel("Quarter")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Observation:** Q4 is the strongest quarter in both years: $210,030 in 2022 and $184,160 in 2023. Q4 represented 46.4% of 2022 revenue and 40.1% of 2023 revenue. However, Q4 revenue declined 12.3% year over year, while Q3 increased 25.3%, so the seasonal peak should not be assumed to grow automatically.

## 7.3 Product Category Performance
The dataset has no individual product identifier, so category-level analysis is the most detailed valid product analysis available.

In [ ]:
category_summary = df_clean.groupby("category").agg(
    revenue=("total_sale", "sum"), transactions=("transactions_id", "count"), quantity_sold=("quantity", "sum"), average_transaction=("total_sale", "mean"), profit=("profit", "sum")
).sort_values("revenue", ascending=False)
category_summary["profit_margin_pct"] = category_summary["profit"] / category_summary["revenue"] * 100
category_summary

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=category_summary.reset_index(), x="category", y="revenue")
plt.title("Revenue by Product Category")
plt.xlabel("Category")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

**Observation:** Electronics generated the highest revenue ($313,810), closely followed by Clothing ($311,070), while Beauty generated $286,840. Clothing had the highest transaction volume (702), while Beauty had the highest average transaction value ($468.69) and profit margin (79.71%). No category is overwhelmingly dominant across all KPIs.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
sns.barplot(data=category_summary.reset_index(), x="category", y="transactions", ax=axes[0])
axes[0].set_title("Transactions by Category")
sns.barplot(data=category_summary.reset_index(), x="category", y="average_transaction", ax=axes[1])
axes[1].set_title("Average Transaction Value")
sns.barplot(data=category_summary.reset_index(), x="category", y="profit_margin_pct", ax=axes[2])
axes[2].set_title("Profit Margin by Category")
plt.tight_layout()
plt.show()

**Observation:** Category performance depends on the KPI used. Clothing leads transaction count and quantity sold; Electronics leads revenue; Beauty leads average transaction value and margin. Volume leadership and value/profitability leadership are therefore not identical.

## 7.4 Customer Demographics — Gender

In [ ]:
gender_summary = df_clean.groupby("gender").agg(
    transactions=("transactions_id", "count"), revenue=("total_sale", "sum"), average_transaction=("total_sale", "mean")
).dropna()
gender_summary["revenue_share_pct"] = gender_summary["revenue"] / gender_summary["revenue"].sum() * 100
gender_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=gender_summary.reset_index(), x="gender", y="revenue", ax=axes[0])
axes[0].set_title("Revenue by Gender")
sns.barplot(data=gender_summary.reset_index(), x="gender", y="average_transaction", ax=axes[1])
axes[1].set_title("Average Transaction Value by Gender")
plt.tight_layout()
plt.show()

**Observation:** Female customers generated 51.05% of revenue ($465,400) from 1,020 transactions, compared with 48.95% ($446,320) from 980 male transactions. Average transaction values were almost identical ($457.62 versus $455.43), so the revenue difference is mainly associated with transaction volume.

## 7.5 Customer Demographics — Age Groups

In [ ]:
age_summary = df_clean.dropna(subset=["age_group"]).groupby("age_group", observed=True).agg(
    customers=("customer_id", "nunique"), transactions=("transactions_id", "count"), revenue=("total_sale", "sum"), average_transaction=("total_sale", "mean")
).reset_index()
age_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=age_summary, x="age_group", y="transactions", ax=axes[0])
axes[0].set_title("Transactions by Age Group")
sns.barplot(data=age_summary, x="age_group", y="average_transaction", ax=axes[1])
axes[1].set_title("Average Transaction Value by Age Group")
plt.tight_layout()
plt.show()

**Observation:** Customers under 25 had the highest average transaction value ($503.92), while customers aged 55–64 had the lowest ($417.82). The 25–34 and 35–44 groups generated the highest revenues ($194,090 and $193,595). These are descriptive differences and do not establish that age causes spending behaviour.

## 7.6 Customer Revenue Concentration

In [ ]:
customer_summary = df_clean.groupby("customer_id").agg(
    transactions=("transactions_id", "count"), revenue=("total_sale", "sum"), average_transaction=("total_sale", "mean")
).sort_values("revenue", ascending=False)
customer_summary["cumulative_revenue_share_pct"] = customer_summary["revenue"].cumsum() / customer_summary["revenue"].sum() * 100
top20_n = max(1, int(np.ceil(len(customer_summary) * 0.20)))
top20_share = customer_summary.head(top20_n)["revenue"].sum() / customer_summary["revenue"].sum() * 100
print(f"Customers analysed: {len(customer_summary):,}")
print(f"Top 20% revenue share: {top20_share:.2f}%")

In [ ]:
plt.figure(figsize=(10, 6))
rank = np.arange(1, len(customer_summary) + 1)
plt.plot(rank, customer_summary["cumulative_revenue_share_pct"])
plt.axvline(top20_n, linestyle="--", label=f"Top 20% ({top20_n:,} customers)")
plt.axhline(80, linestyle="--", label="80% revenue")
plt.title("Cumulative Customer Revenue Contribution")
plt.xlabel("Customers ranked by revenue")
plt.ylabel("Cumulative revenue share (%)")
plt.legend()
plt.tight_layout()
plt.show()

**Observation:** The top 20% of customers account for 44.74% of revenue, not approximately 80%. The dataset therefore does not support describing the customer base as a classic 80/20 revenue distribution.

## 7.7 Correlation Analysis
Correlation measures linear association, not causation. Relationships involving `total_sale`, `quantity`, and `price_per_unit` require caution because total sales is mathematically derived from quantity × unit price.

In [ ]:
correlation_cols = ["age", "quantity", "price_per_unit", "cogs", "total_sale", "profit"]
correlation_matrix = df_clean[correlation_cols].corr()
correlation_matrix

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Matrix of Numerical Variables")
plt.tight_layout()
plt.show()

**Observation:** The strongest linear relationships are between total sale and price per unit (r = 0.85), total sale and COGS (r = 0.71), and total sale and quantity (r = 0.37). Profit is very strongly correlated with total sale (r = 0.98). These relationships are partly structural because total sale is based on quantity × unit price and profit is total sale − COGS; they should not be interpreted as causal effects.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df_clean, x="quantity", y="total_sale", alpha=0.5, ax=axes[0])
axes[0].set_title("Quantity vs. Total Sale")
sns.scatterplot(data=df_clean, x="price_per_unit", y="total_sale", alpha=0.5, ax=axes[1])
axes[1].set_title("Unit Price vs. Total Sale")
plt.tight_layout()
plt.show()

**Observation:** The scatter plots confirm positive relationships between quantity and total sale and between unit price and total sale. The unit-price relationship is stronger, consistent with the direct mathematical role of unit price in transaction revenue.

## 7.8 Additional Insight — What Drives High-Revenue Months?

In [ ]:
monthly_sales.sort_values("revenue", ascending=False).head(10)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
sns.lineplot(data=monthly_sales, x="sale_date", y="transactions", marker="o", ax=axes[0])
axes[0].set_title("Monthly Transaction Volume")
sns.lineplot(data=monthly_sales, x="sale_date", y="average_transaction_value", marker="o", ax=axes[1])
axes[1].set_title("Monthly Average Transaction Value")
plt.tight_layout()
plt.show()

**Observation:** The highest-revenue months are concentrated in September–December. December 2022 was the strongest month at $72,880 and December 2023 reached $69,145. The transaction-count and average-transaction plots help distinguish whether high revenue comes from more purchases, larger baskets, or both.

# 8. Key Findings

### Finding 1 — Strong year-end concentration
Q4 generated $210,030 in 2022 and $184,160 in 2023, making it the strongest quarter in both years. Q4 accounted for 46.4% of 2022 revenue and 40.1% of 2023 revenue. The concentration supports seasonal inventory and staffing planning, while the year-over-year decline shows that Q4 demand should not be assumed to grow automatically.

### Finding 2 — Category leadership depends on the KPI
Electronics led revenue at $313,810, Clothing led transaction volume at 702 transactions, and Beauty recorded the highest average transaction value ($468.69) and profit margin (79.71%). No category dominates every KPI.

### Finding 3 — Customer spending is not a classic Pareto pattern
The top 20% of customers generated 44.74% of revenue. This is materially below the 80% benchmark, so the data does not support calling the customer base a classic 80/20 distribution.

### Finding 4 — Revenue is strongly linked to unit price and structurally linked to profit
Total sale correlates with unit price at 0.85 and with quantity at 0.37. Profit correlates with total sale at 0.98. These results are partly mathematical rather than evidence of causality.

# 9. Business Recommendations

1. **Strengthen seasonal planning:** Prepare inventory, staffing and promotional capacity ahead of the September–December high-demand period, while monitoring year-over-year changes rather than assuming continued growth.
2. **Manage categories using multiple KPIs:** Protect strong revenue categories such as Electronics while investigating why Clothing leads volume and Beauty leads average transaction value and margin.
3. **Use targeted customer segmentation:** Prioritise retention experiments based on actual customer value and purchase behaviour, but do not assume that a small VIP group drives most revenue.
4. **Test age-based campaigns carefully:** Under-25 customers have the highest average transaction value, while 25–44 groups generate the highest total revenue. These differences can inform experiments, not causal conclusions.
5. **Improve data collection:** Add product identifiers, geographic information and richer customer attributes to enable product-level rankings and deeper segmentation.

# 10. Limitations & Conclusion

## Limitations
- The dataset contains category information but no individual product identifier, so a true Top-10-products ranking cannot be produced.
- Customer attributes are limited mainly to age and gender.
- Correlation does not imply causation.
- Three transactions have missing transaction-level numeric values and cannot be reliably reconstructed from the available fields.
- The dataset covers two calendar years, which is useful for observing repeated seasonal patterns but insufficient for robust long-term trend inference.

## Conclusion
The analysis shows a strong year-end sales concentration, relatively balanced gender performance, meaningful differences across age groups, modest category-level variation, and broader customer revenue distribution than a classic Pareto pattern. The most important business implication is to plan for seasonal demand while using multiple KPIs to evaluate categories and customer segments.